# 02 - Exploración y Análisis del Top 10 de Datasets de Ciberseguridad e IA

Este notebook permite cargar, explorar los esquemas e inspeccionar el contenido de los **10 datasets imprescindibles de Ciberseguridad e Inteligencia Artificial** descargados a través de `src/downloader.py`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")

# Directorios de datos
ROOT_DIR = Path.cwd().parent
TOP10_DIR = ROOT_DIR / "data" / "raw" / "top10_security"
MANIFEST_FILE = TOP10_DIR / "manifest.json"

print(f"Directorio de datasets: {TOP10_DIR}")
print(f"¿Existe manifiesto?: {MANIFEST_FILE.exists()}")

## 1. Cargar Manifiesto y Estado de Descargas

In [ ]:
if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    
    df_manifest = pd.DataFrame.from_dict(manifest.get("datasets", {}), orient="index")
    if not df_manifest.empty:
        display(df_manifest[["rank", "category", "size_mb", "file_count", "description"]].sort_values(by="rank"))
    else:
        print("No hay datasets registrados aún en el manifiesto.")
else:
    print("El archivo manifest.json aún no existe. Ejecuta 'python3 -m src.downloader' para iniciar descargas.")

## 2. Inspección del Dataset #1: `jackhhao/jailbreak-classification`
El benchmark de referencia para clasificar intentos de jailbreak y prompts maliciosos en LLMs.

In [ ]:
jailbreak_dir = TOP10_DIR / "01_jackhhao__jailbreak-classification"

if jailbreak_dir.exists():
    # Buscar archivos csv principales
    csv_files = list(jailbreak_dir.glob("**/*.csv"))
    print(f"Archivos CSV encontrados: {[f.name for f in csv_files]}")
    
    sample_file = next((f for f in csv_files if "full" in f.name or "train" in f.name), csv_files[0] if csv_files else None)
    if sample_file:
        df_jb = pd.read_csv(sample_file)
        print(f"\nArchivo cargado: {sample_file.name} | Filas: {len(df_jb)} | Columnas: {list(df_jb.columns)}")
        display(df_jb.head())
else:
    print(f"Directorio {jailbreak_dir.name} no encontrado. Descárgalo con 'python3 -m src.downloader --index 1'")

## 3. Función Auxiliar para Explorar Cualquier Dataset Descargado

In [ ]:
def preview_dataset_folder(folder_name: str, max_rows: int = 5):
    folder = TOP10_DIR / folder_name
    if not folder.exists():
        print(f"Carpeta no encontrada: {folder_name}")
        return
    
    data_files = list(folder.glob("**/*.parquet")) + list(folder.glob("**/*.csv")) + list(folder.glob("**/*.jsonl"))
    print(f"=== Explorando: {folder_name} ===")
    print(f"Total archivos de datos: {len(data_files)}")
    
    if not data_files:
        print("No se encontraron archivos Parquet, CSV ni JSONL.")
        return
    
    for file_path in data_files[:2]:
        print(f"\n Leyendo: {file_path.name} ({file_path.stat().st_size / (1024*1024):.2f} MB)")
        try:
            if file_path.suffix == ".parquet":
                df = pd.read_parquet(file_path)
            elif file_path.suffix == ".csv":
                df = pd.read_csv(file_path)
            elif file_path.suffix == ".jsonl":
                df = pd.read_json(file_path, lines=True)
            
            print(f"   Dimensiones: {df.shape} | Columnas: {list(df.columns)}")
            display(df.head(max_rows))
        except Exception as e:
            print(f"   Error leyendo archivo: {e}")